# Valkey Search Workshop — Streaming Service Demo

## 3-Hour Hands-On Workshop

In this workshop you'll build a movie recommendation system using **Valkey Search**, learning:

1. **Index creation** — TEXT, TAG, NUMERIC, VECTOR fields
2. **FT.SEARCH** — Full-text search, tag filters, numeric ranges
3. **Vector similarity (KNN)** — Find similar movies using embeddings
4. **Hybrid search** — Combine filters with vector similarity
5. **Single-slot indexes** — Sub-ms per-user queries
6. **FT.AGGREGATE** — Platform analytics (trending, most watched)
7. **Cross-index workflows** — Connecting user history to catalog recommendations

### Architecture

| Index | Purpose | Latency |
|:------|:--------|:--------|
| `idx:movies` (Global) | Movie catalog with vectors | Sub-ms to low ms |
| `idx:user:{<id>}:history` (Single-Slot) | One user's watch history | Sub-ms |
| `idx:watch` (Global) | All users' history for analytics | Sub-ms to low ms |

### Dataset
- **10,000 movies** with 768-dim embeddings (from TMDB)
- **60,000 ratings** from 500 users (from MovieLens)


## What is Valkey Search?

[Valkey Search](https://valkey.io/topics/search/) is a high-performance search engine provided as a Valkey module (BSD-3-Clause). It supports:

- **Vector search** — sub-ms to low-ms latency, 99%+ recall
- **Full-text, Tag, and Numeric search** — with complex filters and hybrid queries
- **Real-time indexing** — data is searchable the moment it's written

This makes it ideal for recommendations, semantic search, fraud detection, and any workload that needs low-latency search over operational data.

In this workshop, we use it to build a streaming service with catalog search, personalized recommendations, and platform analytics.

📚 [Documentation](https://valkey.io/topics/search/) | [FT.CREATE reference](https://valkey.io/commands/ft.create/) | [FT.SEARCH reference](https://valkey.io/commands/ft.search/) | [FT.AGGREGATE reference](https://valkey.io/commands/ft.aggregate/)


## Part 1: Setup & Connection (~15 min)

Run the cells below to start Valkey and connect.
Make sure Docker or Podman is installed and running (see README).


In [ ]:
# === SETUP ===
# Starts Valkey with Search module using Docker or Podman.
# Works on Windows (Podman/Docker) and Mac (Podman/Colima/Docker).

import subprocess, socket, time, shutil

# Detect container runtime
runtime = 'podman' if shutil.which('podman') else 'docker'
print(f'Using: {runtime}')

# Stop and remove any existing Valkey container
subprocess.run([runtime, 'rm', '-f', 'valkey'], capture_output=True)
time.sleep(1)

# Start Valkey
subprocess.run([runtime, 'run', '-d', '--name', 'valkey', '-p', '26379:6379',
    'valkey/valkey-bundle:9.1-rc2-alpine',
    'valkey-server', '--save', '', '--protected-mode', 'no'])

print('Waiting for Valkey to start...')
time.sleep(5)

# Verify by pinging inside the container (bypasses Windows networking issues)
result = subprocess.run([runtime, 'exec', 'valkey', 'valkey-cli', 'ping'],
    capture_output=True, text=True)
assert 'PONG' in result.stdout, f'Failed to start. Check: {runtime} logs valkey'
print('✅ Valkey started')


> **Prerequisites**: Docker installed and running.  
> If the cell above fails, run manually:  
> ```
> docker run -d --name valkey -p 6379:6379 valkey/valkey-bundle:9.1.0-rc2 valkey-server --save "" --protected-mode no
> ```

If you are running this workshop on your own machine via Jupyter Notebook, uncomment the following cell and ensure these are installed.

In [ ]:
# %pip install -q valkey pandas numpy

In [ ]:
import os
required = ['catalog.csv', 'ratings.csv']
missing = [f for f in required if not os.path.exists(f'data/{f}')]
if missing:
    print(f'Missing files: {missing}')
    print('Please upload the data/ folder.')
else:
    print(f'\u2713 Data found: {os.listdir("data")}')


In [ ]:
import csv, struct, time, subprocess, json
import pandas as pd
import numpy as np
import valkey

# Connect to Valkey — handle Podman Windows networking
VALKEY_HOST = 'localhost'
VALKEY_PORT = 26379

try:
    r = valkey.Valkey(host=VALKEY_HOST, port=VALKEY_PORT, decode_responses=True, socket_timeout=3)
    r.ping()
except Exception:
    # Podman on Windows: localhost may not route. Get container IP instead.
    inspect = subprocess.run([runtime, 'inspect', 'valkey'], capture_output=True, text=True)
    info = json.loads(inspect.stdout)
    VALKEY_HOST = info[0]['NetworkSettings']['IPAddress'] or 'localhost'
    print(f'localhost failed, using container IP: {VALKEY_HOST}')
    r = valkey.Valkey(host=VALKEY_HOST, port=VALKEY_PORT, decode_responses=True, socket_timeout=3)

r_bin = valkey.Valkey(host=VALKEY_HOST, port=VALKEY_PORT, decode_responses=False, socket_timeout=3)

print(f'Connected: {r.ping()}')
modules = r_bin.module_list()
module_names = [m[b'name'].decode() for m in modules]
print(f'Modules: {module_names}')
assert 'search' in module_names, 'ERROR: search module not loaded!'
print('✅ Ready!')


In [ ]:
# Explore the datasets
import pandas as pd

# --- Two CSVs, Two Document Types ---
#
# catalog.csv  | Source: Remsky/Embeddings__Ultimate_1Million_Movies_Dataset (HuggingFace)
#              |   https://huggingface.co/datasets/Remsky/Embeddings__Ultimate_1Million_Movies_Dataset
#              | One row per movie: metadata + 768-dim embedding (nomic-embed-text on title+tagline+overview)
#              | Columns: tmdb_id, title, overview, genres, tmdb_rating, tmdb_popularity, language, embedding
#              |   tmdb_rating: crowd-sourced movie score from TMDB users (0-10, like IMDb but separate)
#              |   tmdb_popularity: TMDB's external buzz/attention metric (views, watchlists, social)
#              |   These are external signals that complement our own user_rating data from the
#              |   streaming service — e.g., filter catalog by global quality + use our ratings for personalization
#              | Ingested as: HSET movie:{tmdb_id} ...
#              | Index: idx:movies (global)
#
# ratings.csv  | Source: MovieLens 32M
#              |   https://grouplens.org/datasets/movielens/32m/
#              | One row per user-movie rating (pre-joined with movie metadata)
#              | Columns: user_id, title, genres, tmdb_id, user_rating, timestamp
#              | Ingested as: HSET user:{user_id}:watch:{n} ...
#              | Index: idx:user:{id}:history (single-slot) AND idx:watch (global)

print('=== catalog.csv (movie catalog) ===')
catalog = pd.read_csv('data/catalog.csv', nrows=3)
print(f'Columns: {list(catalog.columns)}')
print(catalog.drop(columns=['embedding', 'overview']).to_string(index=False))

print(f'\n=== ratings.csv (user watch history) ===')
ratings = pd.read_csv('data/ratings.csv')
print(f'Rows: {len(ratings)} | Users: {ratings["user_id"].nunique()} | Movies: {ratings["tmdb_id"].nunique()}')
print(ratings.head(3).to_string(index=False))


## Part 2: Global Catalog Index (~45 min)

A **global index** is distributed across all shards in a cluster.
- Each shard holds a portion of the documents
- Queries fan out to every shard, results are merged and returned

**Schema design choices:**
- `title`, `overview` → **TEXT**: tokenized for keyword search and prefix matching
- `genres`, `language` → **TAG**: discrete values, exact/set matching, not tokenized
- `tmdb_rating`, `tmdb_popularity` → **NUMERIC SORTABLE**: range filters and sort ordering
- `vector` → **VECTOR HNSW**: similarity search via KNN ([FT.CREATE docs](https://valkey.io/commands/ft.create/))

We load documents first (plain HSET), then create the index. Valkey backfills existing keys asynchronously.

### 2.1 Load Data


In [ ]:
# Example: what one document looks like in the catalog
# Each movie becomes a HASH key: movie:<tmdb_id>
print('''
Key:    movie:11
Fields:
  title             = "Star Wars"                    (TEXT — full-text searchable)
  overview          = "Princess Leia is captured..." (TEXT — searchable description)
  genres            = "Adventure,Action,Sci-Fi"      (TAG — comma-separated, filterable)
  language = "en"                           (TAG — exact match filter)
  tmdb_rating      = 8.2                            (NUMERIC — range queries, sortable)
  tmdb_popularity        = 101.1                          (NUMERIC — range queries, sortable)
  vector            = <3072 bytes>                   (VECTOR — 768 × float32, for KNN)
''')


In [ ]:
# Start fresh
r.flushall()

# Load movies from CSV with binary-packed vectors
count = 0
pipe = r_bin.pipeline(transaction=False)
t0 = time.time()

with open('data/catalog.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Pack 768 floats into binary blob (required for vector indexing)
        vec_floats = [float(x) for x in row['embedding'].split(',')]
        vec_blob = struct.pack(f'{len(vec_floats)}f', *vec_floats)
        
        pipe.hset(f'movie:{row["tmdb_id"]}'.encode(), mapping={
            b'title': row['title'].encode(),
            b'overview': row['overview'].encode(),
            b'genres': row['genres'].encode(),
            b'language': row['language'].encode(),
            b'tmdb_rating': row['tmdb_rating'].encode(),
            b'tmdb_popularity': row['tmdb_popularity'].encode(),
            b'embedding': vec_blob,
        })
        count += 1
        if count % 500 == 0:
            pipe.execute()
            pipe = r_bin.pipeline(transaction=False)

pipe.execute()
elapsed = time.time() - t0
print(f"✓ Loaded {count} movies in {elapsed:.1f}s ({count/elapsed:.0f} docs/sec)")

### 2.2 Create the Index

Now create the index — Valkey backfills existing keys asynchronously.


In [ ]:
# Drop if exists
try:
    r.execute_command('FT.DROPINDEX', 'idx:movies')
except:
    pass

# Create the global catalog index
r.execute_command('FT.CREATE', 'idx:movies', 'ON', 'HASH', 'PREFIX', '1', 'movie:',
    'SCHEMA',
    'title', 'TEXT',                          # Full-text searchable
    'overview', 'TEXT',                       # Movie description
    'genres', 'TAG', 'SEPARATOR', ',',        # Filterable categories
    'language', 'TAG',               # Language filter
    'tmdb_rating', 'NUMERIC', 'SORTABLE',    # Rating 0-10
    'tmdb_popularity', 'NUMERIC', 'SORTABLE',      # Popularity score
    'embedding', 'VECTOR', 'HNSW', '6',          # Vector similarity
        'TYPE', 'FLOAT32', 'DIM', '768', 'DISTANCE_METRIC', 'COSINE')

print("✓ Index idx:movies created")

In [ ]:
# Wait for backfill to complete
import time
while True:
    info = r.execute_command('FT.INFO', 'idx:movies')
    info_dict = dict(zip(info[::2], info[1::2]))
    pct = float(info_dict.get('backfill_complete_percent', 1.0))
    docs = info_dict.get('num_docs', 0)
    if pct >= 1.0:
        break
    print(f'  Indexing... {pct*100:.0f}% ({docs} docs)', end='\r')
    time.sleep(1)
print(f'✓ Index ready: {info_dict.get("num_docs", 0)} docs indexed')

In [ ]:
# Inspect the index
r.execute_command('FT.INFO', 'idx:movies')


### 2.3 Full-Text Search (FT.SEARCH)

Search movie titles and overviews using natural language text queries.


In [ ]:
# "Find movies about space adventures"
results = r.execute_command('FT.SEARCH', 'idx:movies', 'space adventure',
    'RETURN', '2', 'title', 'genres',
    'LIMIT', '0', '5')

results  # Raw response: [total_matches, key1, [field, val, ...], key2, [...], ...]

In [ ]:
# "Find movies with 'Matrix' in the title"
r.execute_command('FT.SEARCH', 'idx:movies', '@title:Matrix',
    'RETURN', '2', 'title', 'tmdb_rating',
    'LIMIT', '0', '5')

### 2.3 Instant Search & Title Suggestions

Prefix search enables autocomplete / type-ahead as the user types.


In [ ]:
# "User types 'star...' — show title suggestions"
r.execute_command('FT.SEARCH', 'idx:movies', '@title:star*',
    'RETURN', '1', 'title',
    'LIMIT', '0', '5')


In [ ]:
# "User types 'inter...' — show title suggestions"
r.execute_command('FT.SEARCH', 'idx:movies', '@title:inter*',
    'RETURN', '1', 'title',
    'LIMIT', '0', '5')


### 2.4 Tag & Numeric Filters

TAG fields support exact-match filtering. NUMERIC fields support range queries.


In [ ]:
# "Show me all Action movies"
r.execute_command('FT.SEARCH', 'idx:movies', '@genres:{Action}',
    'RETURN', '2', 'title', 'genres',
    'LIMIT', '0', '5')

In [ ]:
# "Top Action movies rated 8 or higher"
r.execute_command('FT.SEARCH', 'idx:movies',
    '@genres:{Action} @tmdb_rating:[8 10]',
    'SORTBY', 'tmdb_popularity', 'DESC',
    'RETURN', '3', 'title', 'tmdb_rating', 'tmdb_popularity',
    'LIMIT', '0', '5')

In [ ]:
# "Highly rated movies that are Comedy or Drama"
r.execute_command('FT.SEARCH', 'idx:movies',
    '@genres:{Comedy | Drama} @tmdb_rating:[8.5 10]',
    'RETURN', '2', 'title', 'genres',
    'LIMIT', '0', '5')

### ✏️ Try it yourself

Find **Horror** movies in **English** with **rating above 7**, sorted by rating descending.


In [ ]:
# Your query here:



### 2.5 Vector Similarity Search (KNN)

Each movie has a 768-dimensional embedding vector. We can find similar movies
using K-Nearest Neighbors (KNN) search with cosine distance.


In [ ]:
# "Find movies similar to Star Wars"
source = r_bin.hgetall(b'movie:11')  # Star Wars (tmdb_id: 11)
print(f"Source movie: {source[b'title'].decode()}")
print(f"Vector size: {len(source[b'embedding'])} bytes ({len(source[b'embedding'])//4} floats)")

# KNN search — find 6 most similar (first is always self, so we skip it)
results = r_bin.execute_command('FT.SEARCH', 'idx:movies',
    '*=>[KNN 6 @embedding $query_vec]',
    'PARAMS', '2', 'query_vec', source[b'embedding'],
    'RETURN', '2', 'title', 'genres',
    'DIALECT', '2')

print()
print('Movies similar to Star Wars:')
for i in range(1, len(results), 2):
    if results[i] == b'movie:11':
        continue
    doc = dict(zip(results[i+1][::2], results[i+1][1::2]))
    print(f"  {doc[b'title'].decode()} ({doc[b'genres'].decode()})")


### 2.6 Hybrid Search (Filter + Vector)

Combine tag/numeric filters with vector similarity — e.g., "Action movies similar to Star Wars".


In [ ]:
# "Action movies similar to Star Wars"
# "Action movies similar to Star Wars"
source_vec = r_bin.hget(b'movie:11', b'embedding')

results = r_bin.execute_command('FT.SEARCH', 'idx:movies',
    '@genres:{Action}=>[KNN 11 @embedding $query_vec]',
    'PARAMS', '2', 'query_vec', source_vec,
    'RETURN', '3', 'title', 'genres', 'tmdb_rating',
    'DIALECT', '2')

print('Action movies similar to Star Wars:')
for i in range(1, len(results), 2):
    if results[i] == b'movie:11':
        continue
    doc = dict(zip(results[i+1][::2], results[i+1][1::2]))
    title = doc[b'title'].decode()
    rating = doc[b'tmdb_rating'].decode()
    print(f'  {title} ({rating}\u2605)')


### ✏️ Try it yourself

Write a query to find **Animation** movies with **tmdb_rating above 8**, sorted by tmdb_popularity.


In [ ]:
# Your query here:



## Part 3: Single-Slot Index — Per-User History (~30 min)

### Cluster Topology: Global vs Single-Slot

In a Valkey cluster, keys are distributed across shards by hash slot.

**Global index** (Part 2): keys spread across all shards. Every query fans out.

**Single-slot index**: all keys share the same hash slot → same shard.
- Query executes locally on one shard — no fanout, no merge
- Sub-ms
- One index per user — ephemeral (create on login, drop on logout)
- Achieved by key prefix: `user:{1}:watch:*` — the `{1}` hash tag pins all keys to the same slot

```
┌────────────────┐  ┌────────────────┐  ┌────────────────┐
│    Shard 1     │  │    Shard 2     │  │    Shard 3     │
│                │  │                │  │                │
│  movie:*       │  │  movie:*       │  │  movie:*       │  ← Global: partitioned
│                │  │                │  │                │
│  user:{1}:*    │  │                │  │  user:{2}:*    │  ← Single-slot: pinned
└────────────────┘  └────────────────┘  └────────────────┘
```

### 3.1 Load & Create Per-User Indexes


In [ ]:
# Example: what one document looks like in a user's watch history
# Each rating becomes a HASH key: user:{<id>}:watch:<n>
print('''
Key:    user:{1}:watch:0
Fields:
  title     = "Toy Story"           (TEXT — searchable)
  genres    = "Adventure,Animation" (TAG — filterable)
  user_rating = 4.0                   (NUMERIC — user's user's rating, 0.5-5.0)
  timestamp = 944249077             (NUMERIC — when they rated it)
  tmdb_id   = "862"                 (TAG — links to movie:862 in global catalog)
''')


In [ ]:
# Load ALL user ratings into Valkey, then create single-slot indexes for 3 demo users
ratings_df = pd.read_csv('data/ratings.csv')
print(f'Loading {len(ratings_df)} ratings from {ratings_df["user_id"].nunique()} users...')

# Load all data: user:{id}:watch:{n} keys
t0 = time.time()
user_doc_counts = {}
for _, row in ratings_df.iterrows():
    uid = int(row['user_id'])
    n = user_doc_counts.get(uid, 0)
    user_doc_counts[uid] = n + 1
    r.hset(f'user:{{{uid}}}:watch:{n}', mapping={
        'title': str(row['title']),
        'genres': str(row['genres']),
        'user_rating': str(row['user_rating']),
        'timestamp': str(int(row['timestamp'])),
        'tmdb_id': str(int(row['tmdb_id'])),
    })
elapsed = time.time() - t0
print(f'\u2713 Loaded {len(ratings_df)} docs in {elapsed:.1f}s')

# Create single-slot indexes for 3 demo users only
demo_users = list(user_doc_counts.keys())[:3]
print(f'\nCreating single-slot indexes for demo users: {demo_users}')
for user_id in demo_users:
    idx_name = f"idx:user:{{{user_id}}}:history"
    prefix = f"user:{{{user_id}}}:watch:"
    try:
        r.execute_command('FT.DROPINDEX', idx_name)
    except:
        pass
    r.execute_command('FT.CREATE', idx_name, 'ON', 'HASH',
        'PREFIX', '1', prefix,
        'SCHEMA',
        'title', 'TEXT',
        'genres', 'TAG', 'SEPARATOR', ',',
        'user_rating', 'NUMERIC', 'SORTABLE',
        'timestamp', 'NUMERIC', 'SORTABLE',
        'tmdb_id', 'TAG')
    print(f'  idx:user:{{{user_id}}}:history \u2014 {user_doc_counts[user_id]} docs')

print('\n\u2713 All data loaded + 3 single-slot indexes created')


In [ ]:
# Inspect one user's index
r.execute_command('FT.INFO', f'idx:user:{{{demo_users[0]}}}:history')


### 3.2 Query User History

In [ ]:
user_id = demo_users[0]
idx = f"idx:user:{{{user_id}}}:history"

# "What did I watch most recently?"
print(f'--- User {user_id}: Recent watches ---')
results = r.execute_command('FT.SEARCH', idx, '@timestamp:[-inf +inf]',
    'SORTBY', 'timestamp', 'DESC',
    'RETURN', '3', 'title', 'user_rating', 'timestamp',
    'LIMIT', '0', '5')
print(f'Total: {results[0]} movies')
for i in range(1, len(results), 2):
    doc = dict(zip(results[i+1][::2], results[i+1][1::2]))
    print(f"  {doc['title']} - {doc['user_rating']}\u2605")


In [ ]:
# "What are my favorite Action movies?"
print(f'--- User {user_id}: Top Action movies ---')
results = r.execute_command('FT.SEARCH', idx,
    '@genres:{Action} @user_rating:[4 5]',
    'SORTBY', 'user_rating', 'DESC',
    'RETURN', '2', 'title', 'user_rating',
    'LIMIT', '0', '5')
print(f'{results[0]} matches')
for i in range(1, len(results), 2):
    doc = dict(zip(results[i+1][::2], results[i+1][1::2]))
    print(f"  {doc['title']} - {doc['user_rating']}\u2605")


In [ ]:
# "Have I watched Star Wars?"
print(f'--- User {user_id}: Have I watched Star Wars? ---')
results = r.execute_command('FT.SEARCH', idx, '@title:Star Wars',
    'RETURN', '2', 'title', 'user_rating',
    'LIMIT', '0', '5')
if results[0] > 0:
    for i in range(1, len(results), 2):
        doc = dict(zip(results[i+1][::2], results[i+1][1::2]))
        print(f"  Yes! {doc['title']} - rated {doc['user_rating']}\u2605")
else:
    print('  No - not watched yet.')


### ✏️ Try it yourself

Query your user's single-slot index: find all **Drama** movies they rated **3.5 or higher**.


In [ ]:
# Your query here (use the idx variable from above):



## Part 4: Global User Index & FT.AGGREGATE (~30 min)

We already loaded per-user watch history as `user:{<id>}:watch:<n>` keys.
Now we create a **second index** over those same keys — but as a **global** index.

- Same data, no duplication — just a different index with `PREFIX 1 user:`
- Because it's global, queries fan out to all shards and aggregate across all users
- Enables cross-user analytics: trending, most watched, avg ratings
- Queried with `FT.AGGREGATE` — server-side GROUPBY, REDUCE, SORT

Single-slot answers "what did *I* watch?" — this index answers "what is *everyone* watching?"

### 4.1 Create the Global Index (no new data needed)


In [ ]:
# The data already exists — same user:{<id>}:watch:<n> keys from Part 3.
# We just create a new global index over them with a broader prefix.
#
# Per-user index:  PREFIX 1 user:{1}:watch:   (only user 1's keys)
# Global index:    PREFIX 1 user:           (ALL users' keys)
#
# Same documents, two indexes, two access patterns.

try:
    r.execute_command('FT.DROPINDEX', 'idx:watch')
except:
    pass

r.execute_command('FT.CREATE', 'idx:watch', 'ON', 'HASH', 'PREFIX', '1', 'user:',
    'SCHEMA',
    'title', 'TEXT',
    'genres', 'TAG', 'SEPARATOR', ',',
    'user_rating', 'NUMERIC', 'SORTABLE',
    'timestamp', 'NUMERIC', 'SORTABLE',
    'tmdb_id', 'TAG')

# Wait for backfill
import time
while True:
    info = r.execute_command('FT.INFO', 'idx:watch')
    d = dict(zip(info[::2], info[1::2]))
    if float(d.get('backfill_complete_percent', 1.0)) >= 1.0:
        break
    time.sleep(0.5)
print(f"\u2713 idx:watch ready: {d.get('num_docs', 0)} docs indexed (same keys, global view)")


In [ ]:
# Inspect the global user index
r.execute_command('FT.INFO', 'idx:watch')


In [ ]:
# "What are the most watched movies on the platform?"
results = r.execute_command('FT.AGGREGATE', 'idx:watch', '@user_rating:[-inf +inf]',
    'LOAD', '1', '@tmdb_id',
    'GROUPBY', '1', '@tmdb_id',
    'REDUCE', 'COUNT', '0', 'AS', 'watch_count',
    'SORTBY', '2', '@watch_count', 'DESC',
    'LIMIT', '0', '10')

# Raw response
print("Raw:", results[:3], "...\n")

# Server computed the counts — we just look up titles
print("Most Watched:")
for row in results[1:]:
    doc = dict(zip(row[::2], row[1::2]))
    title = r.hget(f"movie:{doc['tmdb_id']}", 'title') or doc['tmdb_id']
    print(f"  {title} — {doc['watch_count']} watches")

In [ ]:
# "Highest average rating across users, only including movies rated by at least 50 people"
results = r.execute_command('FT.AGGREGATE', 'idx:watch', '@user_rating:[-inf +inf]',
    'LOAD', '2', '@tmdb_id', '@user_rating',
    'GROUPBY', '1', '@tmdb_id',
    'REDUCE', 'AVG', '1', '@user_rating', 'AS', 'avg_rating',
    'REDUCE', 'COUNT', '0', 'AS', 'num_ratings',
    'FILTER', '@num_ratings >= 50',
    'SORTBY', '2', '@avg_rating', 'DESC',
    'LIMIT', '0', '10')

for row in results[1:]:
    doc = dict(zip(row[::2], row[1::2]))
    title = r.hget(f"movie:{doc['tmdb_id']}", 'title') or doc['tmdb_id']
    print(f"  {title} — avg {float(doc['avg_rating']):.2f}★ ({doc['num_ratings']} ratings)")

In [ ]:
# "What genres are most popular?"
results = r.execute_command('FT.AGGREGATE', 'idx:watch', '@user_rating:[-inf +inf]',
    'LOAD', '1', '@genres',
    'GROUPBY', '1', '@genres',
    'REDUCE', 'COUNT', '0', 'AS', 'watch_count',
    'SORTBY', '2', '@watch_count', 'DESC',
    'LIMIT', '0', '10')

for row in results[1:]:
    doc = dict(zip(row[::2], row[1::2]))
    print(f"  {doc['genres']} — {doc['watch_count']} watches")

### ✏️ Try it yourself

Write an FT.AGGREGATE query on `idx:watch` to find the **most loved Action movies** —
movies in the Action genre that have been rated **4-5★ by at least 5 users**.
Sort by number of fans descending.

Hint: filter with `@genres:{Action} @user_rating:[4 5]`, then GROUPBY `@tmdb_id`, REDUCE COUNT, FILTER, SORTBY.


In [ ]:
# Solution: Most loved Action movies (rated 4-5★ by 5+ users)
results = r.execute_command('FT.AGGREGATE', 'idx:watch', '@genres:{Action} @user_rating:[4 5]',
    'LOAD', '1', '@tmdb_id',
    'GROUPBY', '1', '@tmdb_id',
    'REDUCE', 'COUNT', '0', 'AS', 'num_fans',
    'FILTER', '@num_fans >= 5',
    'SORTBY', '2', '@num_fans', 'DESC',
    'LIMIT', '0', '10')

print('--- Most Loved Action Movies (5+ fans rated 4-5★) ---')
for row in results[1:]:
    doc = dict(zip(row[::2], row[1::2]))
    tmdb_id = doc.get('tmdb_id', '')
    title = r.hget(f'movie:{tmdb_id}', 'title') or tmdb_id
    print(f'  {title} — {doc["num_fans"]} fans')


## Part 5: "Because You Watched..." — Recommendation Flow (~30 min)

This is the end-to-end recommendation pattern. All three indexes work together:

1. **Single-slot** → find what the user likes (sub-ms)
2. **Global catalog** → find similar content via vector KNN (sub-ms to low ms)
3. **Global user index** → blend with trending data (sub-ms to low ms)

This powers experiences like "Because you watched Inception..." on streaming platforms.

### 5.1 The Pipeline


In [ ]:
user_id = demo_users[0]
idx = f"idx:user:{{{user_id}}}:history"

print(f"=== Because You Watched... (User {user_id}) ===\n")

# Step 1: What did they recently love? (single-slot, sub-ms)
print('Step 1: Find most recent highly-rated movie (single-slot)')
recent = r.execute_command('FT.SEARCH', idx,
    '@user_rating:[4 5]',
    'SORTBY', 'timestamp', 'DESC',
    'RETURN', '3', 'title', 'genres', 'tmdb_id',
    'LIMIT', '0', '1')
seed = dict(zip(recent[2][::2], recent[2][1::2]))
seed_title = seed['title']
seed_genre = seed['genres'].split(',')[0].strip()
seed_tmdb = seed['tmdb_id']
print(f"  -> '{seed_title}' (genre: {seed_genre}, tmdb_id: {seed_tmdb})\n")

# Step 2: Fetch its vector from global catalog (key lookup)
print(f'Step 2: Fetch vector for movie:{seed_tmdb} (key lookup)')
vec_blob = r_bin.hget(f'movie:{seed_tmdb}'.encode(), b'embedding')
print(f'  -> {len(vec_blob)} bytes ({len(vec_blob)//4} floats)\n')

# Step 3: KNN — find similar movies in that genre (global, sub-ms to low ms)
print(f'Step 3: "Because you watched {seed_title}, you might like..." (global KNN)')
knn = r_bin.execute_command('FT.SEARCH', 'idx:movies',
    f'@genres:{{{seed_genre}}}=>[KNN 6 @embedding $q]',
    'PARAMS', '2', 'q', vec_blob,
    'RETURN', '3', 'title', 'genres', 'tmdb_rating',
    'DIALECT', '2')

for i in range(1, len(knn), 2):
    if f'movie:{seed_tmdb}'.encode() == knn[i]:
        continue  # skip the source movie itself
    d = dict(zip(knn[i+1][::2], knn[i+1][1::2]))
    print(f'  {d[b"title"].decode()} ({d[b"tmdb_rating"].decode()}\u2605)')


## ✏️ Part 6: Exercises (~30 min)

### Exercise 1: Filter by language
Find French (`fr`) comedies rated above 7. Hint: use `@language:{fr}`.

### Exercise 2: User dedup
Before recommending a movie, check if the user already watched it using their single-slot index.

### Exercise 3: Trending + Personal
Combine FT.AGGREGATE (trending movies) with the user's single-slot (exclude already watched).

### Exercise 4: Custom aggregation
Write an FT.AGGREGATE query to find the average rating per genre across all users.


In [ ]:
# Exercise 1: Multi-language category search
# Find French comedies rated above 7.
# Hints:
#   - Language field: @language:{fr}
#   - Genre field: @genres:{Comedy}
#   - Rating field: @tmdb_rating:[7 10]

# YOUR CODE HERE


In [ ]:
# Exercise 2: User dedup
# Before recommending a movie, check if the user already watched it.
# Hints:
#   - Use the single-slot index: idx:user:{user_id}:history
#   - Search by tmdb_id: @tmdb_id:{<id>}
#   - If result[0] > 0, they've seen it

# YOUR CODE HERE


In [ ]:
# Exercise 3: Trending + Personal
# Get trending movies from idx:watch, then exclude ones the user already watched.
# Hints:
#   - FT.AGGREGATE idx:watch to get top movies by watch_count
#   - For each result, FT.SEARCH the user's single-slot index by @tmdb_id:{id}
#   - Only show movies where the user hasn't seen it (result[0] == 0)

# YOUR CODE HERE


In [ ]:
# Exercise 4: Average rating per genre
# Use FT.AGGREGATE on idx:watch to compute average rating per genre.
# Hints:
#   - LOAD the @genres and @user_rating fields
#   - GROUPBY 1 @genres
#   - REDUCE AVG 1 @user_rating AS avg_rating
#   - REDUCE COUNT 0 AS count
#   - FILTER @count >= 10

# YOUR CODE HERE


## Summary

### Reusable Patterns

| Pattern | How | When |
|:--------|:----|:-----|
| Instant search / autocomplete | `@title:prefix*` | Search bars, type-ahead |
| Faceted browse | `@genre:{X} @user_rating:[min max]` | Catalog filtering |
| "More like this" | Vector KNN with pre-filter | Recommendations |
| "Because you watched..." | Single-slot → vector → dedup | Personalization |
| Trending / analytics | FT.AGGREGATE GROUPBY + REDUCE | Dashboards, feeds |
| Per-user data | Single-slot index | Session state, preferences |

### Latency & Tuning

- **FLAT vs HNSW**: FLAT for <10K docs (exact, fast ingest). HNSW for >10K (approximate, fast query).
- **Single-slot vs Global**: Use single-slot for user-scoped data. Global for shared data.
- **SORTABLE**: Add to NUMERIC fields you sort by — avoids runtime sort overhead.
- **RETURN**: Only return fields you need — reduces network payload.
- **LIMIT**: Always paginate — don't fetch all matches.
- **Load before index**: For bulk ingestion, HSET first then FT.CREATE (backfill is async).

### What You Built

```
Global Catalog (idx:movies)      → Browse, search, vector similarity
Single-Slot (idx:user:{X}:history)  → Per-user, sub-ms, ephemeral
Global Users (idx:watch)          → Cross-user analytics, trending
```
